# 06 Signal-Family Ablation

This notebook runs and reviews validation-only signal-family ablations using the fixed square-root-weighted MSResCNN-MLP recipe. Each ablation removes both raw channels and engineered summary features for the selected signal families.

## Setup

The setup cell resolves the repository root, imports the reusable helpers, and defines the output directory used by the ablation workflow.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.signal_ablation import (
    DEFAULT_STAGE17_OUTPUT_DIR,
    assert_validation_only_stage17_outputs,
    default_stage17_signal_ablation_specs,
    load_stage17_signal_ablation_summary,
    run_stage17_signal_ablation,
    summarize_stage17_signal_ablation,
)
from src.train import TrainConfig

stage17_train_feature_path = repo_root / "data" / "processed" / "features_train.csv"
stage17_validation_feature_path = repo_root / "data" / "processed" / "features_val.csv"
stage17_output_dir = repo_root / DEFAULT_STAGE17_OUTPUT_DIR
stage17_base_config = TrainConfig(
    raw_dir=repo_root / "data" / "raw",
    epoch_index_path=repo_root / "data" / "interim" / "epoch_index.csv",
    preprocessing_metadata_path=(
        repo_root / "data" / "processed" / "preprocessing_metadata.json"
    ),
    participant_array_cache_dir=repo_root / "data" / "processed" / "deep" / "participants",
    train_feature_path=stage17_train_feature_path,
    validation_feature_path=stage17_validation_feature_path,
    feature_preprocessing_metadata_path=(
        repo_root / "data" / "processed" / "feature_preprocessing_metadata.json"
    ),
)
figures_dir = stage17_output_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

## Ablation Design

This cell lists the planned ablations. Signal families are grouped as cardiovascular (`BVP`, `HR`, and `IBI`), movement (`ACC`), electrodermal activity (`EDA`), and temperature (`TEMP`). For each run, the helper applies the same family selection to both raw channels and engineered feature columns.

In [ ]:
stage17_specs = default_stage17_signal_ablation_specs()
spec_table = pd.DataFrame(
    [
        {
            "ablation_id": spec.name,
            "label": spec.label,
            "included_families": ", ".join(spec.included_families),
            "omitted_families": ", ".join(spec.omitted_families) or "none",
            "description": spec.description,
        }
        for spec in stage17_specs
    ]
)
display(spec_table)

## Run Or Load Results

The run switch controls whether the ablations are trained. Leave it disabled when reviewing existing artifacts. When enabled, the helper writes ablated feature tables, per-run outputs, and a combined summary under `results/stage17_signal_ablation`. Completed ablations are skipped by default when their saved summary matches the current fixed recipe.

In [ ]:
RUN_STAGE17_SIGNAL_ABLATION = False
SKIP_COMPLETED_ABLATIONS = True
OVERWRITE_ABLATED_FEATURE_TABLES = False

if RUN_STAGE17_SIGNAL_ABLATION:
    missing_inputs = [
        path
        for path in [stage17_train_feature_path, stage17_validation_feature_path]
        if not path.exists()
    ]
    if missing_inputs:
        raise FileNotFoundError(
            "Stage 17 requires engineered train/validation feature tables: "
            f"{missing_inputs}"
        )
    stage17_summary = run_stage17_signal_ablation(
        stage17_specs,
        base_config=stage17_base_config,
        output_dir=stage17_output_dir,
        train_feature_path=stage17_train_feature_path,
        validation_feature_path=stage17_validation_feature_path,
        overwrite_feature_tables=OVERWRITE_ABLATED_FEATURE_TABLES,
        skip_completed=SKIP_COMPLETED_ABLATIONS,
    )
else:
    stage17_summary = load_stage17_signal_ablation_summary(stage17_output_dir)
    if stage17_summary.empty:
        print(
            "No Stage 17 summary found yet. Set RUN_STAGE17_SIGNAL_ABLATION = True "
            "to run the validation-only ablation workflow."
        )

if not stage17_summary.empty:
    assert_validation_only_stage17_outputs(stage17_output_dir)
    stage17_summary = summarize_stage17_signal_ablation(stage17_summary)
    display(stage17_summary)

## Validation Metric Deltas

This table focuses on the primary validation metrics and their differences from the full signal-family run.

In [ ]:
if stage17_summary.empty:
    print("Stage 17 summary is not available yet.")
else:
    metric_columns = [
        "ablation_id",
        "ablation_label",
        "macro_f1",
        "delta_macro_f1",
        "balanced_accuracy",
        "delta_balanced_accuracy",
        "accuracy",
        "delta_accuracy",
    ]
    metric_columns = [column for column in metric_columns if column in stage17_summary.columns]
    metric_table = stage17_summary[metric_columns]
    if "macro_f1" in metric_table.columns:
        metric_table = metric_table.sort_values("macro_f1", ascending=False)
    display(metric_table)

## Macro F1 By Ablation

This figure compares validation macro F1 across signal-family ablations and saves the result as `stage17_macro_f1_by_ablation.png`.

In [ ]:
if stage17_summary.empty or "macro_f1" not in stage17_summary.columns:
    print("Macro F1 summary is not available yet.")
else:
    plot_data = stage17_summary.sort_values("macro_f1", ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(
        data=plot_data,
        x="macro_f1",
        y="ablation_label",
        hue="ablation_label",
        dodge=False,
        legend=False,
        palette="viridis",
        ax=ax,
    )
    ax.set_xlabel("Validation macro F1")
    ax.set_ylabel("")
    ax.set_title("Stage 17 Signal-Family Ablation")
    fig.tight_layout()
    output_path = figures_dir / "stage17_macro_f1_by_ablation.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print(f"Saved {output_path}")

## Metric Drop From Full Recipe

This figure shows validation macro-F1 change relative to the full signal-family recipe and saves the result as `stage17_delta_macro_f1.png`.

In [ ]:
if stage17_summary.empty or "delta_macro_f1" not in stage17_summary.columns:
    print("Delta macro F1 summary is not available yet.")
else:
    plot_data = stage17_summary[stage17_summary["ablation_id"] != "full"].copy()
    plot_data = plot_data.sort_values("delta_macro_f1", ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["#b23b3b" if value < 0 else "#2f7f5f" for value in plot_data["delta_macro_f1"]]
    ax.barh(plot_data["ablation_label"], plot_data["delta_macro_f1"], color=colors)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Delta validation macro F1 vs full recipe")
    ax.set_ylabel("")
    ax.set_title("Signal-Family Contribution")
    fig.tight_layout()
    output_path = figures_dir / "stage17_delta_macro_f1.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print(f"Saved {output_path}")

## Per-Class F1 Deltas

This heatmap shows how each ablation changes class-specific validation F1 relative to the full recipe and saves the result as `stage17_class_f1_deltas.png`.

In [ ]:
class_delta_columns = ["delta_Wake_f1", "delta_Non_REM_f1", "delta_REM_f1"]
if stage17_summary.empty or not set(class_delta_columns).issubset(stage17_summary.columns):
    print("Class-specific F1 deltas are not available yet.")
else:
    heatmap_data = stage17_summary[stage17_summary["ablation_id"] != "full"].copy()
    heatmap_data = heatmap_data.set_index("ablation_label")[class_delta_columns]
    heatmap_data = heatmap_data.rename(
        columns={
            "delta_Wake_f1": "Wake",
            "delta_Non_REM_f1": "Non-REM",
            "delta_REM_f1": "REM",
        }
    )
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".3f",
        center=0,
        cmap="vlag",
        linewidths=0.5,
        ax=ax,
    )
    ax.set_xlabel("Class")
    ax.set_ylabel("")
    ax.set_title("Class F1 Change vs Full Recipe")
    fig.tight_layout()
    output_path = figures_dir / "stage17_class_f1_deltas.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print(f"Saved {output_path}")

## Training Curves

This panel reviews validation macro-F1 trajectories for completed ablation runs and saves the result as `stage17_training_curves.png`.

In [ ]:
history_path = stage17_output_dir / "all_history.csv"
if not history_path.exists():
    print("Stage 17 training history is not available yet.")
else:
    history = pd.read_csv(history_path)
    if history.empty or "macro_f1" not in history.columns:
        print("Stage 17 training history does not contain macro F1 values.")
    else:
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.lineplot(
            data=history,
            x="epoch",
            y="macro_f1",
            hue="ablation_label",
            marker="o",
            ax=ax,
        )
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Validation macro F1")
        ax.set_title("Stage 17 Training Curves")
        ax.legend(title="Ablation", bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()
        output_path = figures_dir / "stage17_training_curves.png"
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        print(f"Saved {output_path}")